<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/torneos/notebooks/c6_l2.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C6-L2 · Spearman y MMC
Spearman por era y aporte sobre el consenso.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/torneos/data/c6_l2.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c6_l2.csv'), Path('data/c6_l2.csv'), Path('c6_l2.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)
print(df.head(5).to_string(index=False))


In [ ]:
# Spearman por era = Pearson sobre rangos, promediado
por_era = df.groupby('era').apply(lambda g: g['pred'].rank().corr(g['actual'].rank()), include_groups=False)
print(por_era.round(4).to_string())
score = float(por_era.mean())
print(f'Score medio (CORR)={score:.4f}  dispersión={por_era.std():.4f}')
assert por_era.notna().all() and np.isfinite(score)
assert score > 0, 'se espera filo positivo en esta demo'
assert ((por_era >= -1) & (por_era <= 1)).all()


In [ ]:
# Global vs por era: la global esconde la inconsistencia
glob = df['pred'].rank().corr(df['actual'].rank())
print(f'Global={glob:.4f} vs media por era={score:.4f}')
assert np.isfinite(glob)


In [ ]:
# MMC: correlación con el residuo tras quitar el consenso
df['residuo'] = df['actual'] - df['mm_pred']
mmc_era = df.groupby('era').apply(lambda g: g['pred'].rank().corr(g['residuo'].rank()), include_groups=False)
mmc = float(mmc_era.mean())
copia = df.groupby('era').apply(lambda g: g['mm_pred'].rank().corr(g['residuo'].rank()), include_groups=False).mean()
print(f'MMC señal={mmc:.4f}  MMC copia del consenso={copia:.4f}')
assert mmc > 0, 'la señal propia debe aportar sobre el consenso'
assert abs(copia) < 0.2, 'la copia del consenso no debe aportar'



In [ ]:
# Chequeo automático L2
assert df['era'].nunique() == 12 and len(df) == 60
assert np.isfinite(score) and np.isfinite(mmc)
print(f'OK L2: CORR={score:.4f} MMC={mmc:.4f} verificados')
